In [ ]:
import json
import os
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

In [ ]:
def load_results(path, prefix, model):
    digits = set([str(i) for i in range(10)])
    prefix_len = len(prefix)

    files = os.listdir(path)
    relevant_files = []
    for file_name in files:
        if ".json" in file_name and file_name.find(prefix) == 0 and file_name[prefix_len] in digits:
            end_ind = file_name.find(".json")
            val = float(file_name[prefix_len:end_ind])
            relevant_files.append((val, file_name))
    relevant_files = sorted(relevant_files)
    results = []
    for val, file_name in relevant_files:
        with open(f"{path}/{file_name}", "r") as fp:
            results.append(json.load(fp))
    return results

In [ ]:
def extract_value(results, model, metric, min_ndcg=-1, ind=-1):
    data = [np.array(result[model][metric]) for result in results]
    if ind != -1:
        data = [dat[:,ind] for dat in data]
    if min_ndcg != -1:
        ndcg = [np.array(result[model]["NDCG"]) for result in results]
        masks = [val > min_ndcg for val in ndcg]
        data = [dat[masks[i]] for i,dat in enumerate(data)]
    return [np.mean(dat) for dat in data]

In [ ]:
def combine_results(models, metrics):
    results = {}
    for model, model_results in models:
        results[model] = {}
        for metric in metrics:
            ind = 0 if metric == "Rep AUC" else -1
            results[model][metric] = extract_value(model_results, model, metric, ind=ind)
    return results

In [ ]:
def plot_scatter_params(results, models, metrics, auc=False):
    fig = go.Figure()
    markers = ["circle", "x", "star", "square"]
    # Add traces
    for i, model in enumerate(models):
        for j, metric in enumerate(metrics):
            fig.add_trace(go.Scatter(x=results[model]["Rep AUC"], y=results[model][metric],
                mode='markers',
                marker={"symbol": markers[j]},
                name=f"{models[i]} {metric}"))
    #fig.update_xaxes(range = [0.0,1.0])
    #fig.update_yaxes(range = [0.3,1.1])
    fig.add_vline(x=0.5)
    if auc:
        fig.update_layout(yaxis_title="Recommendation AUC")
        fig.add_hline(y=0.5)
    fig.update_yaxes(
        scaleanchor="x",
        scaleratio=1,
    )
    fig.update_layout(xaxis_title="Representation AUC")
    fig.update_layout(
        legend=dict(
            x=0.02,
            y=.95,
        )
    )
    fig.show()

In [ ]:
def plot_scatter_multi(results_list, models, metrics, n_rows, n_cols, titles, y_axis="Recommendation AUC", auc=False, aliases=None,ly=0.45, uy=1.0, lx=0.5,ux=1.0):
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=titles)
    markers = ["circle", "x", "star", "square"]
    color_list = px.colors.qualitative.Plotly
    flat_ind = 0
    for col in range(1,n_cols+1):
        for row in range(1, n_rows+1):
            # Add traces
            results = results_list[flat_ind]
            local_ind = 0
            for i, model in enumerate(models):
                for j, metric in enumerate(metrics):
                    model_name = models[i]
                    metric_name = metric
                    if aliases is not None and model in aliases[0]:
                        model_name = aliases[0][model_name]
                    if aliases is not None and metric in aliases[1]:
                        metric_name = aliases[1][metric_name]
                    fig.add_trace(go.Scatter(x=results[model]["Rep AUC"], y=results[model][metric],
                        mode='markers',
                        marker={"symbol": markers[j], "color": color_list[local_ind]},
                        name=f"{model_name} {metric_name}",
                        showlegend=flat_ind == 0),
                        row=row,
                        col=col,
                    )
                    local_ind += 1
            #fig.update_xaxes(range = [0.0,1.0])
            #fig.update_yaxes(range = [0.3,1.1])
            fig.add_vline(x=0.5, row=row, col=col)
            if auc:
                fig.add_hline(y=0.5, row=row, col=col)
            fig.update_yaxes(title_text=y_axis)#,scaleanchor="x",scaleratio=1,range=[ly,ux],row=row,col=col)
            fig.update_xaxes(title_text="Representation AUC")#, range=[lx,ux], row=row, col=col)
            flat_ind += 1
    for i in range(flat_ind):
        ax_no = i+1 if i!= 0 else ""
        y_range=[lx,ux] if auc else [-1,0.85]
        args = {f"xaxis{ax_no}":dict(range=[lx,ux]), f"yaxis{ax_no}":dict(range=y_range)}
        fig.update_layout(**args)
    fig.show()

In [ ]:
def plot_scatter_multi2(results, models, metrics_list, n_rows, n_cols, titles, y_axis="Recommendation AUC", auc=False, aliases=None,ly=0.45, uy=1.0, lx=0.5,ux=1.0):
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=titles)
    markers = ["circle", "x", "star", "square"]
    color_list = px.colors.qualitative.Plotly
    flat_ind = 0
    for col in range(1,n_cols+1):
        for row in range(1, n_rows+1):
            # Add traces
            metrics = metrics_list[flat_ind]
            local_ind = 0
            for i, model in enumerate(models):
                for j, metric in enumerate(metrics):
                    model_name = models[i]
                    metric_name = metric
                    if aliases is not None and model in aliases[0]:
                        model_name = aliases[0][model_name]
                    if aliases is not None and metric in aliases[1]:
                        metric_name = aliases[1][metric_name]
                    fig.add_trace(go.Scatter(x=results[model]["Rep AUC"], y=results[model][metric],
                        mode='markers',
                        marker={"symbol": markers[j], "color": color_list[local_ind]},
                        name=f"{model_name}",
                        showlegend=flat_ind == 0),
                        row=row,
                        col=col,
                    )
                    local_ind += 1
            #fig.update_xaxes(range = [0.0,1.0])
            #fig.update_yaxes(range = [0.3,1.1])
            fig.add_vline(x=0.5, row=row, col=col)
            if auc:
                fig.add_hline(y=0.5, row=row, col=col)
                fig.update_yaxes(title_text=y_axis)#,scaleanchor="x",scaleratio=1,range=[ly,ux],row=row,col=col)
            else:
                fig.update_yaxes(title_text=y_axis[flat_ind],row=row,col=col)
            fig.update_xaxes(title_text="Representation AUC")#, range=[lx,ux], row=row, col=col)
            flat_ind += 1
    for i in range(flat_ind):
        if not auc:
            break
        ax_no = i+1 if i!= 0 else ""
        y_range=[lx,ux] if auc else [-1,0.85]
        args = {f"xaxis{ax_no}":dict(range=[lx,ux]), f"yaxis{ax_no}":dict(range=y_range)}
        fig.update_layout(**args)
    fig.show()

In [ ]:
model_aliases = {"BASE": "VAE", "PERTURB": "VAERel", "POP ": "POP", "RAND": "RAND", "INFO": "VAEAfrl*", "UNFAIR": "Dem. POP", "UNFAIR_DIV": "Max Division"}
metric_aliases = {"Rep AUC": "Rep. AUC", "R_Med_AUC": "Dem. Ratio AUC", "AUC": "Neural AUC", "Item_ratio": "Item ratio"}
aliases = [model_aliases, metric_aliases]

In [ ]:
path = "movielens"
perturb_results = load_results(path, "perturb_dub_G", "PERTURB")
perturb_old_results = load_results(path, "results", "PERTURB")
info_results = load_results(path, "results_lambda", "INFO")

metrics = ["Rep AUC", "R_Med_AUC10", "R_Med_AUC", "AUC", "Item_ratio", "Kendall-Tau", "NDCG"]
adv_models = [("PERTURB", perturb_results), ("INFO", info_results)]
rel_models = [("PERTURB", perturb_old_results), ("INFO", info_results)]
model_names = [x[0] for x in rel_models]

rel_resultsG = combine_results(rel_models, metrics)
adv_resultsG = combine_results(adv_models, metrics)

perturb_results = load_results(path, "perturb_dub_A", "PERTURB")
perturb_old_results = load_results(path, "perturb_age", "PERTURB")
info_results = load_results(path, "info_age", "INFO")

metrics = ["Rep AUC", "R_Med_AUC10", "R_Med_AUC", "AUC", "Item_ratio", "Kendall-Tau", "NDCG"]
adv_models = [("PERTURB", perturb_results), ("INFO", info_results)]
rel_models = [("PERTURB", perturb_old_results), ("INFO", info_results)]
model_names = [x[0] for x in rel_models]

rel_resultsA = combine_results(rel_models, metrics)
adv_resultsA = combine_results(adv_models, metrics)

In [ ]:
 # GENDER

In [ ]:
plot_scatter_params(adv_resultsG, model_names, ["R_Med_AUC", "AUC"], auc=True)

In [ ]:
plot_scatter_params(adv_resultsG, model_names, ["Item_ratio"])

In [ ]:
plot_scatter_params(adv_resultsG, model_names, ["Kendall-Tau"])

In [ ]:
plot_scatter_params(adv_resultsG, model_names, ["NDCG"])

In [ ]:
plot_scatter_params(adv_resultsA, model_names, ["R_Med_AUC", "AUC"], auc=True)

In [ ]:
plot_scatter_params(adv_resultsA, model_names, ["Item_ratio"])

In [ ]:
plot_scatter_params(adv_resultsA, model_names, ["Kendall-Tau"])

In [ ]:
plot_scatter_params(adv_resultsA, model_names, ["NDCG"])

In [ ]:
plot_scatter_multi([rel_resultsG, rel_resultsA], model_names, ["R_Med_AUC", "AUC"], 1, 2, ["Gender", "Age", "Gender", "Age"], auc=True, aliases=aliases, lx=0.47, ux=0.84)

In [ ]:
adv_aliases = [dict(aliases[0]), dict(aliases[1])]
adv_aliases[0]["PERTURB"] = "VAE2adv"
plot_scatter_multi([adv_resultsG, adv_resultsA], model_names, ["R_Med_AUC", "AUC"], 1, 2, ["Gender", "Age", "Gender", "Age"], auc=True, aliases=adv_aliases, lx=0.47, ux=0.83)

In [ ]:
# SYNTH

In [ ]:
path = "grid_tests/Perturb_Synth_05"
perturb_results = load_results(path, "perturb", "PERTURB")
path = "grid_tests/Info_Synth_05"
info_results = load_results(path, "info", "INFO")

metrics = ["Rep AUC", "R_Med_AUC10", "R_Med_AUC", "AUC", "Item_ratio", "Kendall-Tau", "NDCG"]
models = [("PERTURB", perturb_results), ("INFO", info_results)]
model_names = [x[0] for x in models]

results = {}
for model, model_results in models:
    results[model] = {}
    for metric in metrics:
        ind = 0 if metric == "Rep AUC" else -1
        results[model][metric] = extract_value(model_results, model, metric, ind=ind)

In [ ]:
plot_scatter_params(results, model_names, ["R_Med_AUC", "AUC"], auc=True)

In [ ]:
plot_scatter_params(results, model_names, ["Item_ratio"])

In [ ]:
eps_vals = [5,62,74]
eps_results = []
for eps_val in eps_vals:
    
    path = f"grid_tests/Perturb_Synth_0{eps_val}"
    perturb_results = load_results(path, "perturb", "PERTURB")
    path = f"grid_tests/Info_Synth_0{eps_val}"
    info_results = load_results(path, "info", "INFO")
    
    metrics = ["Rep AUC", "R_Med_AUC10", "R_Med_AUC", "AUC", "Item_ratio", "Kendall-Tau", "NDCG"]
    models = [("PERTURB", perturb_results), ("INFO", info_results)]
    model_names = [x[0] for x in models]
    
    results = {}
    for model, model_results in models:
        results[model] = {}
        for metric in metrics:
            ind = 0 if metric == "Rep AUC" else -1
            results[model][metric] = extract_value(model_results, model, metric, ind=ind)
    eps_results.append(results)
plot_scatter_multi(eps_results, model_names, ["R_Med_AUC", "AUC"], 1, 3, ["ε=0.5", "ε=0.62", "ε=0.74"], auc=True, aliases=aliases, lx=0.47, ux=0.9)

In [ ]:
#path = "grid_tests/n_u4000_new/"
#path = "grid_tests/eps4000_new/"
path = "grid_tests/eps_hires/"
path2 = "grid_tests/eps_hires_both/"

In [ ]:
def get_model_results(paths):
    param_vals = []
    model_results = []
    for path in paths:
        folders = os.listdir(path)
        folder_params = sorted([(float(x), x) for x in folders])
        param_vals += [x[0] for x in folder_params]
        for param, name in folder_params:
            with open(f"{path}{name}/results.json", "r") as fp:
                model_results.append(json.load(fp))
    return model_results, param_vals

In [ ]:
model_results, param_vals = get_model_results([path])
multi_results, _ = get_model_results([path, path2])

In [ ]:
metrics = ["R_Med_AUC", "AUC", "Item_ratio", "Kendall-Tau", "NDCG", "Rep AUC"]
models = ["POP ", "RAND", "UNFAIR", "UNFAIR_DIV", "BASE", "PERTURB"] #, "INFO"]

In [ ]:
results = {}
multi = {}
for model in models:
    results[model] = {}
    multi[model] = {}
    for metric in metrics:
        if metric in model_results[0][model]:
            ind = 0 if metric == "Rep AUC" else -1
            results[model][metric] = extract_value(model_results, model, metric, ind=ind)
        if metric in multi_results[0][model]:
            ind = 0 if metric == "Rep AUC" else -1
            multi[model][metric] = extract_value(multi_results, model, metric, ind=ind)

In [ ]:
plot_scatter_params(multi, ["BASE", "PERTURB"], ["AUC"], auc=True)

In [ ]:
plot_scatter_params(multi, ["BASE", "PERTURB"], ["R_Med_AUC"], auc=True)

In [ ]:
plot_scatter_params(multi, ["BASE", "PERTURB"], ["Kendall-Tau"])

In [ ]:
plot_scatter_multi2(multi, ["BASE", "PERTURB"], [["AUC"],["R_Med_AUC"]], 1, 2, ["Neural AUC", "Dem. Ratio AUC"], auc=True, aliases=aliases, lx=0.45, ux=1.01)

In [ ]:
plot_scatter_multi2(multi, ["BASE", "PERTURB"], [["Item_ratio"],["Kendall-Tau"]], 1, 2, ["(minimize)", "(maximize)"], auc=False, y_axis=["Item Ratio","Kendall-Tau"],aliases=aliases, lx=0.45, ux=1.01)

In [ ]:
def plot_models(x, results, metric, models, x_label, aliases=None):
    n_models = len(models)
    fig = go.Figure()

    # Add traces
    for i, model in enumerate(models):
        name = models[i]
        if aliases is not None and model in aliases[0]:
            name = aliases[0][name]
        fig.add_trace(go.Scatter(x=x, y=results[model][metric],
            mode='lines',
            name=name))
    # fig.update_xaxes(range = [0.0,1.0])
    #fig.update_yaxes(range = [0.3,1.1])
    if aliases is not None and metric in aliases[1]:
        metric = aliases[1][metric]
    fig.update_layout(yaxis_title=metric)
    fig.update_layout(xaxis_title=x_label)
    fig.show()

In [ ]:
def plot_models_all(x, results, metrics, models, x_label, aliases=None):
    n_models = len(models)
    n_rows = len(metrics)
    fig = make_subplots(rows=n_rows, cols=1,
                    shared_xaxes=True,
                    vertical_spacing=0.02)
    
    color_list = px.colors.qualitative.Plotly
    ind = 0
    for row in range(1,n_rows+1):
        # Add traces
        metric = metrics[ind]
        for i, model in enumerate(models):
            name = models[i]
            if aliases is not None and model in aliases[0]:
                name = aliases[0][name]
            fig.add_trace(go.Scatter(x=x, y=results[model][metric],
                mode='lines',
                name=name,
                line={"color":color_list[i]},
                showlegend=ind==0),
                row=row,
                col=1)
        # fig.update_xaxes(range = [0.0,1.0])
        #fig.update_yaxes(range = [0.3,1.1])
        if aliases is not None and metric in aliases[1]:
            metric = aliases[1][metric]
        fig.update_yaxes(title_text=metric, row=row, col=1)
        ind += 1
    fig.update_xaxes(title_text=x_label, row=n_rows,col=1)
    
    fig.update_layout(
        height=1000,
    )
    fig.show()

In [ ]:
def plot_compare(x, results, metrics, models, x_label, y_label, scale=False, aliases=None):
    fig = go.Figure()

    # Add traces
    for i, model in enumerate(models):
        for metric in metrics:
            y = results[model][metric]
            if metric == "Kendall-Tau" and scale:
                y = -0.5*(np.array(y)-1)

            model_name = models[i]
            metric_name = metric
            if aliases is not None and model in aliases[0]:
                model_name = aliases[0][model_name]
            if aliases is not None and metric in aliases[1]:
                metric_name = aliases[1][metric_name]
            fig.add_trace(go.Scatter(x=x, y=y,
                mode='lines',
                name=f"{model_name} {metric_name}"))
    #fig.update_xaxes(range = [0.0,1.0])
    #fig.update_yaxes(range = [0.3,1.1])
    fig.update_layout(xaxis_title=x_label)
    fig.update_layout(yaxis_title=y_label)
    fig.show()

In [ ]:
plot_models(param_vals, results, "Kendall-Tau", models, "Epsilon", aliases)

In [ ]:
plot_models(param_vals, results, "AUC", models, "Epsilon", aliases)

In [ ]:
plot_models(param_vals, results, "R_Med_AUC", models, "Epsilon", aliases)

In [ ]:
plot_models(param_vals, results, "Item_ratio", models, "Epsilon", aliases)

In [ ]:
plot_models(param_vals, results, "NDCG", models, "Epsilon", aliases)

In [ ]:
plot_compare(param_vals, results, ["AUC", "R_Med_AUC"], ["BASE", "PERTURB"], "ε", "Rec. AUC", aliases=aliases)

In [ ]:
plot_compare(param_vals, results, ["Item_ratio", "Kendall-Tau"], ["POP ", "RAND"], "Epsilon", "Kendal/Item ratio", aliases=aliases)

In [ ]:
plot_models_all(param_vals, results, ["AUC", "R_Med_AUC", "Item_ratio", "Kendall-Tau"], models, "ε", aliases)

In [ ]:
path = "grid_tests/n_u4000_new/"
model_results, param_vals = get_model_results([path])
metrics = ["R_Med_AUC", "AUC", "Item_ratio", "Kendall-Tau", "NDCG"]
models = ["POP ", "RAND", "UNFAIR", "UNFAIR_DIV", "BASE", "PERTURB"]
results = {}
for model in models:
    results[model] = {}
    for metric in metrics:
        ind = 0 if metric == "Rep AUC" else -1
        results[model][metric] = extract_value(model_results, model, metric, ind=ind)

In [ ]:
plot_models(param_vals, results, "Kendall-Tau", models, "#Users", aliases)

In [ ]:
plot_models(param_vals, results, "AUC", models, "#Users", aliases)

In [ ]:
plot_models(param_vals, results, "R_Med_AUC", models, "#Users", aliases)

In [ ]:
plot_models(param_vals, results, "Item_ratio", models, "#Users", aliases)

In [ ]:
plot_models_all(param_vals, results, ["AUC", "R_Med_AUC", "Item_ratio", "Kendall-Tau"], models, "#Users", aliases)

In [ ]:
path = "grid_tests/skewdness/"
model_results, param_vals = get_model_results([path])
metrics = ["R_Med_AUC", "AUC", "Item_ratio", "Kendall-Tau", "NDCG"]
models = ["POP ", "RAND", "UNFAIR", "UNFAIR_DIV", "BASE", "PERTURB"]
results = {}
for model in models:
    results[model] = {}
    for metric in metrics:
        ind = 0 if metric == "Rep AUC" else -1
        results[model][metric] = extract_value(model_results, model, metric, ind=ind)

In [ ]:
plot_models(param_vals, results, "Kendall-Tau", models, "Minority ratio", aliases)

In [ ]:
plot_models(param_vals, results, "AUC", models, "Minority ratio", aliases)

In [ ]:
plot_models(param_vals, results, "R_Med_AUC", models, "Minority ratio", aliases)

In [ ]:
plot_models(param_vals, results, "Item_ratio", models, "Minority ratio", aliases)

In [ ]:
plot_compare(param_vals, results, ["Item_ratio", "Kendall-Tau"], ["BASE", "PERTURB"], "Minority ratio", "Neg.Norm. Kendall-Tau/Item ratio", aliases=aliases, scale=True)

In [ ]:
plot_models_all(param_vals, results, ["AUC", "R_Med_AUC", "Item_ratio", "Kendall-Tau"], models, "Minority ratio", aliases)